In [ ]:
############################################## SETTINGS ##############################################  
# settings
sim = 'sim1001'  # current steps for firing curve control
# sim = 'sim1002'  # current steps for firing curve 20% KCNQ
# sim = 'sim1003'  # current steps for firing curve control exact rheobase
# sim = 'sim1004'  # current steps for firing curve 20% KCNQ exact rheobase

cell_type='ispn'
model = 3

import os, sys
# compute absolute path of main folder
cwd = os.getcwd()

if os.path.exists(os.path.join(cwd, 'settings.py')):
    main_dir = cwd
else:
    main_dir = os.path.abspath(os.path.join(cwd, '..'))

# set CWD or add to path
os.chdir(main_dir)
sys.path.insert(0, main_dir)

# then import/run settings
%run -i settings.py

from analysis_functions import *

# Get the current working directory
current_wd = os.getcwd()
# 'sim' + str(sim)

downsample = True

home = os.path.expanduser('~')

# Set working directory
external = False
external_name = 'MacOS10'
target = f'model{model}'

if not external:
    if downsample:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'
else:
    if downsample:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'


# create path to directory to save analysis
base_path = os.path.join(home, 'Documents', 'Repositories', 'msNEURON_Belal2026', 'analysis')
sim_image_path = os.path.join(base_path, cell_type, sim)
os.makedirs(sim_image_path, exist_ok=True)

save = False

In [ ]:
# calculate surface area of a the neuron
# note NEURON’s default spatial units are micrometres (µm) so area returns µm²

cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=None)
area, total_length = surface_area(cell=cell, spines=spines)

# draw cell

cell_coordinates = []

for sec in cell.somalist:
    h('access ' + sec.name())
    x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
    cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])

# Setup for dendritic sections
for sec in cell.dendlist:
    for seg in sec:
        x, y, z, diam = interpolate_3d(sec, seg.x)
        cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])
    
cell_coordinates = np.array(cell_coordinates, dtype=object)

    
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=0.8, s=2, color='grey', height=600, width=600, plane='xy', mirror=False)

fig_morphology.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    fig_morphology.write_image(f"{sim_image_path}/morphology.svg", format='svg', scale=1)


In [ ]:
# use load_data_dicts to load data
sims_dir = os.path.join(wd, sim)
data_dict = load_data_dicts(wd, sim, cell_type, verbose=True)

In [ ]:
# check memory usage
report_memory(verbose=True)

In [ ]:
# basic experimental metadata
ds = data_dict['metadata']['ds']
dt = data_dict['metadata']['dt']
dt = ds * dt

sim_time = data_dict['metadata']['sim_time']
step_end = sim_time - 200
step_duration = 2000
step_start = step_end - step_duration
step_current = -10

current_step_range = list(range(80, 401, 10)) 

mechs2extract = ['kaf', 'kas', 'kir', 'kcnq', 'sk', 'bk']

print(step_start, step_duration, step_end)


In [ ]:
# generate plots
x = np.arange(0,len(data_dict['vsoma'][0])*dt, dt)
_range_subset = [i for i in range(0, len(current_step_range), 2)]

fig = plot9(x=x,ydict=data_dict['vsoma'], _range=current_step_range, _range_subset=_range_subset,
            yabline=[-85, -60], xaxis_range=[step_start-100, sim_time], yaxis_range=[-95,40], ds=1)
fig.show(config={"toImageButtonOptions": {"format": "svg"}})

if save:
    os.makedirs(sim_image_path, exist_ok=True)
    fig.write_image(f"{sim_image_path}/fig_summary1.svg", format='svg')

In [ ]:
# generate plots
x = np.arange(0,len(data_dict['vsoma'][0])*dt, dt)
if cell_type == 'ispn':
    _range_subset=[3, 6, 12]
else:
    _range_subset=[3, 9, 12]   
    

fig = plot9(x=x,ydict=data_dict['vsoma'], _range=current_step_range, _range_subset=_range_subset,
            yabline=[-85, -60], xaxis_range=[step_start-100, sim_time], yaxis_range=[-95,40], ds=1)

fig.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    sim_image_path = '{}/model{}/{}/images/sim{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim)
    os.makedirs(sim_image_path, exist_ok=True)
    fig.write_image(f"{sim_image_path}/fig_summary1.svg", format='svg', scale=3)

In [ ]:
for mech in mechs2extract:
    plot_mech_current(mech=mech, data_dict=data_dict, _range=current_step_range, _range_subset=_range_subset, ds=1,
                      sim_image_path=sim_image_path, sim_time=sim_time, step_start=step_start, dt=dt, save=save)

In [ ]:
from scipy.optimize import least_squares

ydict = data_dict['vsoma']
dt_spike = deltat if 'deltat' in globals() else dt

spike_counts = [count_spikes(np.asarray(trace).squeeze(), dt=dt_spike) for trace in ydict]

I = np.asarray(current_step_range, float)
spikes = np.asarray(spike_counts, float)
rate = spikes / float(step_duration)
def mod(p, x):
    I0, rmax, I50, n = p
    z = np.clip(x - I0, 0, None)
    return rmax * (z**n) / (z**n + I50**n + 1e-12)

def resid(p):
    yhat = mod(p, I)
    return (yhat - rate)

I_pos = I[rate > 0]
I0_init = np.percentile(I_pos, 10) if I_pos.size else np.min(I)
rmax_init = max(rate.max(), 1.0)
I50_init = max(np.percentile(np.clip(I - I0_init, 0, None), 60), 1e-6)
n_init = 2.0
p0 = np.array([I0_init, rmax_init, I50_init, n_init])
lb = np.array([np.min(I)-50.0, 0.0, 1e-6, 0.5])
ub = np.array([np.max(I), rate.max()*10 + 100.0, (np.max(I)-np.min(I))*5 + 1000.0, 6.0])

res = least_squares(resid, p0, bounds=(lb, ub), max_nfev=2000)
p = res.x

I_fit = np.linspace(I.min(), I.max(), 300)
rate_fit = mod(p, I_fit)

lw = 1
plot_color = 'grey'

fig = go.Figure()
fig.add_trace(go.Scatter(x=I, y=spikes, mode='markers', marker=dict(size=6, color=plot_color), name='Spikes'))
fig.add_trace(go.Scatter(x=I_fit, y=rate_fit*step_duration, mode='lines', line=dict(color=plot_color, width=lw), name='Hill fit'))
fig.update_layout(
    title=dict(
        text='Spike Count vs Current Step',
        x=0.5,
        xanchor='center',
        font=dict(color=plot_color, size=12)
    ),
    xaxis=dict(title='Current Step (pA)', 
               showgrid=False, 
               zeroline=False, 
               ticks='outside',
               showline=True, 
               mirror=False, 
               linecolor=plot_color,
               tickcolor=plot_color,
               linewidth=lw, 
               tickwidth=lw),
    yaxis=dict(title='Spike Count', 
               showgrid=False,
               zeroline=False, 
               ticks='outside',
               showline=True, 
               mirror=False, 
               linecolor=plot_color,
               tickcolor=plot_color,
               linewidth=lw, 
               tickwidth=lw,
               range=[-5, 100]),
    template='plotly_white', 
    font=dict(family='Calibri', size=10, color=plot_color),
    width=500, 
    height=400,
    paper_bgcolor='rgba(0,0,0,0)', 
    plot_bgcolor='rgba(0,0,0,0)'
)
fig.update_traces(line_simplify=False)

fig.show(config={"toImageButtonOptions": {"format": "svg"}})

if save:
    fig.write_image(f"{sim_image_path}/fig_spike_count_vs_steps_summary.svg", format='svg', scale=3) 

In [ ]:
# pull out somatic conductances from json
import json

if cell_type == 'dspn':
    params='params_dMSN3.json'
elif cell_type == 'ispn':
    params='params_iMSN3.json'     

with open(params, 'r') as f:
    params_dict = json.load(f)

gbar = {}
for k in ['kaf', 'kas', 'kir', 'kcnq', 'pas']:
    key = f"gbar_{k}_somatic" if k != 'pas' else 'g_pas_all'
    val = float(params_dict[key]['Value'])
    sf = scale_factors.get(k, 1) if 'scale_factors' in locals() and isinstance(scale_factors, dict) else 1
    gbar[k] = val * sf

def kaf_g(v, gbar):
    a_m = 1.5 / (1 + np.exp((v - 5) / -17))
    b_m = 0.6 / (1 + np.exp((v - 11) / 9))
    m_inf = a_m / (a_m + b_m)
    a_h = 0.105 / (1 + np.exp((v + 120) / 22))
    b_h = 0.065 / (1 + np.exp((v + 54) / -11))
    h_inf = a_h / (a_h + b_h)
    return gbar * (m_inf**2) * h_inf

def kas_g(v, gbar):
    a_m = 0.25 / (1 + np.exp((v - 50) / -20))
    b_m = 0.05 / (1 + np.exp((v + 90) / 35))
    m_inf = a_m / (a_m + b_m)
    a_h = 0.0025 / (1 + np.exp((v + 95) / 16))
    b_h = 0.002 / (1 + np.exp((v - 50) / -70))
    h_inf = 0.2 + (a_h / (a_h + b_h)) * 0.8
    return gbar * (m_inf**2) * h_inf

def kir_g(v, gbar):
    m_inf = 1 / (1 + np.exp((v + 102) / 13))
    return gbar * m_inf

def kcnq_g(v, gbar):
    m_inf = 1 / (1 + np.exp((-59.5 - v) / 10.3))
    return gbar * (m_inf**4)

v_rest = -85
v_test = -60
Cm = 180e-12
funcs = {'kaf': kaf_g, 'kas': kas_g, 'kir': kir_g, 'kcnq': kcnq_g}

g_rest = {k: f(v_rest, gbar[k]) for k, f in funcs.items()}
g_rest['pas'] = gbar['pas']
total_g_rest = sum(g_rest.values())
Rin_rest = 1 / (total_g_rest * 1e-4) / 1e6

g_60 = {k: f(v_test, gbar[k]) for k, f in funcs.items()}
g_60['pas'] = gbar['pas']
total_g_60 = sum(g_60.values())
Rin_60 = 1 / (total_g_60 * 1e-4) / 1e6

v_range = np.linspace(-90, -40, 200)
Rin_vals = []
for v in v_range:
    g_tot = sum([f(v, gbar[k]) for k, f in funcs.items()]) + gbar['pas']
    Rin_vals.append(1 / (g_tot * 1e-4) / 1e6)

V_th = -59.5
ΔV_rest = V_th - v_rest
lo, hi = sorted([v_rest, V_th])
mask = (v_range >= lo) & (v_range <= hi)
Rin_path = np.array(Rin_vals)[mask]
Rin_peak = float(np.max(Rin_path))
Rin_at_th = float(np.interp(V_th, v_range, Rin_vals))

plot_color = 'gray'

layout_common = dict(
    width=560, height=340,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=60, r=160, t=20, b=50)
)

lw = 1
ymin = min(Rin_vals)
ymax = max(Rin_vals)
xrange_common = [-100, -40]

# Fig 1 Input resistance vs Vm
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=v_range, y=Rin_vals,
    mode='lines',
    line=dict(color=plot_color, width=lw),
    name='Rin'
))
fig1.add_trace(go.Scatter(
    x=[V_th, V_th],
    y=[ymin, ymax],
    mode='lines',
    line=dict(color=plot_color, dash='dot', width=lw),
    name='V_th'
))
fig1.update_layout(
    **layout_common,
    xaxis=dict(title='Membrane potential (mV)', range=xrange_common,
               showgrid=False, ticks='outside',
               mirror=False, linecolor=plot_color, tickcolor=plot_color,
               showline=True, zeroline=False),
    yaxis=dict(title='Input resistance (MΩ)', range=[ymin, ymax],
               showgrid=False, ticks='outside',
               mirror=False, linecolor=plot_color, tickcolor=plot_color,
               showline=True, zeroline=False),
    showlegend=False,
    font=dict(family='Calibri', size=10, color=plot_color)
)
fig1.update_traces(line_simplify=False)
fig1.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    fig1.write_image(f"{sim_image_path}/fig_input_resistance_vs_Vm_summary.svg", format='svg', scale=3)

# Fig 2 Conductance vs Vm (semilog)
cols = palette_cols('berlin', len(funcs), alpha=0.8)
semilog = True
fig2 = go.Figure()
for (name, func), col in zip(funcs.items(), cols):
    fig2.add_trace(go.Scatter(
        x=v_range, y=func(v_range, gbar[name]),
        mode='lines',
        line=dict(color=col, width=lw),
        name=name
    ))

xaxis_style = dict(
    title='Membrane potential (mV)',
    range=xrange_common,
    showgrid=False,
    ticks='outside',
    mirror=False,
    linecolor=plot_color,
    tickcolor=plot_color,
    showline=True,
    zeroline=False
)
yaxis_style = dict(
    title='Conductance (S/cm²)',
    showgrid=False,
    ticks='outside',
    mirror=False,
    linecolor=plot_color,
    tickcolor=plot_color,
    showline=True,
    zeroline=False
)
if semilog:
    major_ticks = [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0]
    minor_ticks = []
    for i in range(len(major_ticks) - 1):
        base = major_ticks[i]
        minor_ticks += [base * f for f in range(2, 10)]
    yaxis_style.update(
        type='log',
        tickvals=major_ticks,
        ticktext=[f"10<sup>{int(np.log10(val))}</sup>" for val in major_ticks],
        minor=dict(tickvals=minor_ticks, ticks='outside', showgrid=False),
        showexponent='none'
    )
fig2.update_layout(
    **layout_common,
    xaxis=xaxis_style,
    yaxis=yaxis_style,
    showlegend=True,
    legend=dict(borderwidth=0, x=1.05, y=1, xanchor='left', yanchor='top',
                font=dict(color=plot_color)),
    font=dict(family='Calibri', size=10, color=plot_color)
)

fig2.update_traces(line_simplify=False)
fig2.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    fig2.write_image(f"{sim_image_path}/fig_conductance_vs_Vm_summary.svg", format='svg', scale=3)

If linear-scale plot looks exponential on a semilog axis (i.e. straight line on log scale), that means the conductance changes exponentially with voltage — usually indicating strong voltage dependence, which can correspond to either inward or outward rectification depending on direction.

In [ ]:
# create a html version
if save:
    import os
    output_dir = os.path.join('docs', cell_type)
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    !jupyter nbconvert "passive tune analysis.ipynb" --to html --output-dir="{output_dir}"
